# XGBoost — Fraud Detection with Calibration

Same data as the LR notebook. We'll see:
1. XGBoost's gain over the linear baseline.
2. Why raw GBM probabilities are **miscalibrated** and how to fix it.
3. Early stopping and feature importance.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

SEED = 42
df = pd.read_parquet(Path('..') / 'data' / 'transactions.parquet')

# XGBoost handles categoricals natively; mark them as 'category' dtype.
for col in ['device_type', 'country']:
    df[col] = df[col].astype('category')

X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, stratify=y_trainval, random_state=SEED)
print(f"train: {len(X_train):,}  val: {len(X_val):,}  test: {len(X_test):,}")

## Train with `scale_pos_weight` + early stopping

`scale_pos_weight = N_neg / N_pos` upweights the rare class in the loss. Early stopping on a validation set is the fastest way to regularize a GBM.

In [ ]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {spw:.2f}")

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='aucpr',
    early_stopping_rounds=30,
    random_state=SEED,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"best iteration: {model.best_iteration}")

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC : {average_precision_score(y_test, y_proba):.4f}")
print(f"Brier  : {brier_score_loss(y_test, y_proba):.6f}  (lower is better)")

## Calibration: why GBM probabilities lie

`scale_pos_weight` distorts predicted probabilities. The model is **ranking** well (good AUC) but a predicted '0.7' is not really a 70% chance of fraud. If a downstream policy uses those probabilities as expected-loss inputs, this matters.

Fix: fit an isotonic or Platt scaler on a held-out fold.

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy='quantile')
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'k--', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='raw XGBoost')
plt.xlabel('Predicted probability')
plt.ylabel('Empirical fraud rate')
plt.title('Reliability diagram (before calibration)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Re-train without scale_pos_weight, then calibrate.
base = XGBClassifier(
    n_estimators=model.best_iteration + 1,
    max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', enable_categorical=True,
    random_state=SEED,
)
calibrated = CalibratedClassifierCV(base, method='isotonic', cv=3)
calibrated.fit(X_trainval, y_trainval)

y_proba_cal = calibrated.predict_proba(X_test)[:, 1]
print(f"ROC-AUC (cal): {roc_auc_score(y_test, y_proba_cal):.4f}")
print(f"PR-AUC  (cal): {average_precision_score(y_test, y_proba_cal):.4f}")
print(f"Brier   (cal): {brier_score_loss(y_test, y_proba_cal):.6f}")

frac_pos, mean_pred = calibration_curve(y_test, y_proba_cal, n_bins=10, strategy='quantile')
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'k--', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='isotonic-calibrated')
plt.xlabel('Predicted probability'); plt.ylabel('Empirical fraud rate')
plt.title('Reliability diagram (after calibration)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Feature importance

Use `gain` — average loss reduction when a feature is used in a split. Not perfect (biased toward high-cardinality features), but a fast sanity check.

Note: For interview-grade explanations, SHAP is the right tool. Gain is shown here for brevity.

In [ ]:
imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
imp.head(10).plot.barh(figsize=(6, 4)).invert_yaxis()
plt.title('XGBoost feature importance (gain)'); plt.show()

**Takeaways**
- XGBoost should beat the LR baseline on PR-AUC; if it doesn't, suspect leakage or bad features.
- Always calibrate before treating outputs as probabilities for policy decisions.
- The noise features should rank near the bottom of importance. If they don't, retune `max_depth` / `min_child_weight` or regularize harder.